In [ ]:
import gc
import weakref
import eqiora
from IPython.display import display

def make_mesh():
    graph = eqiora.geometry.CadAuthoredGraph.rectangle_extrusion(
        x_bounds=(0.0, 2.2), y_bounds=(0.0, 0.41), plane_z=0.0, depth=1.0,
        modeling_tolerance=1e-10,
    ).circular_through_cut(
        center=(0.2, 0.2), radius=0.05, boolean_tolerance=1e-10,
    )
    geometry = graph.planar_section(named_topology={
        'fluid': graph.face_handle('end-cap'),
        'inlet': graph.face_handle('profile-x-lower'),
        'outlet': graph.face_handle('profile-x-upper'),
        'walls': (graph.face_handle('profile-y-lower'),
                  graph.face_handle('profile-y-upper')),
        'cylinder': graph.face_handle('cut-wall'),
    })
    request = eqiora.meshing.MeshRequest(
        maximum_boundary_error=1e-4, minimum_mean_ratio=1e-5,
        maximum_boundary_facets=50,
    )
    plan = eqiora.meshing.resolve(geometry, request)
    return eqiora.meshing.generate(geometry, plan=plan)

def capture(mesh):
    coordinates = mesh.coordinates
    cells = mesh.cells
    return {
        'text': repr(mesh),
        'source_digest': mesh.source_digest,
        'realized_geometry_digest': mesh.realized_geometry_digest,
        'mesh_digest': mesh.digest,
        'correspondence_digest': mesh.correspondence_digest,
        'realization_digest': mesh.realization_digest,
        'canonical_bytes': mesh.canonical_bytes,
        'coordinates': coordinates,
        'coordinate_bytes': coordinates.tobytes(order='C'),
        'coordinate_shape': coordinates.shape,
        'coordinate_dtype': coordinates.dtype.str,
        'coordinate_writeable': coordinates.flags.writeable,
        'cells': cells,
        'cell_bytes': cells.tobytes(order='C'),
        'cell_shape': cells.shape,
        'cell_dtype': cells.dtype.str,
        'cell_writeable': cells.flags.writeable,
    }

def assert_unchanged(mesh, expected):
    assert repr(mesh) == expected['text']
    assert mesh.source_digest == expected['source_digest']
    assert mesh.realized_geometry_digest == expected['realized_geometry_digest']
    assert mesh.digest == expected['mesh_digest']
    assert mesh.correspondence_digest == expected['correspondence_digest']
    assert mesh.realization_digest == expected['realization_digest']
    assert mesh.canonical_bytes == expected['canonical_bytes']
    assert mesh.coordinates is expected['coordinates']
    assert mesh.coordinates.tobytes(order='C') == expected['coordinate_bytes']
    assert mesh.coordinates.shape == expected['coordinate_shape']
    assert mesh.coordinates.dtype.str == expected['coordinate_dtype']
    assert mesh.coordinates.flags.writeable is expected['coordinate_writeable']
    assert mesh.cells is expected['cells']
    assert mesh.cells.tobytes(order='C') == expected['cell_bytes']
    assert mesh.cells.shape == expected['cell_shape']
    assert mesh.cells.dtype.str == expected['cell_dtype']
    assert mesh.cells.flags.writeable is expected['cell_writeable']

WIDGET_VIEW_MIME = 'application/vnd.jupyter.widget-view+json'

def current_delegate_model_id(target_mesh):
    # `interfaces.python-rich-mesh-display` evidence O7: the kernel names
    # the close target from the Mesh object
    # it holds. The display hook reuses the open delegate, so this returns the
    # current delegate's model id without any browser-supplied value.
    bundle = target_mesh._repr_mimebundle_(include={WIDGET_VIEW_MIME})
    return bundle[WIDGET_VIEW_MIME]['model_id']

def close_delegate(model_id):
    from ipywidgets import Widget
    Widget.widgets[model_id].close()

main_close_runs = 0
temporary_close_runs = 0

mesh = make_mesh()
accepted_snapshot = capture(mesh)

In [ ]:
mesh

In [ ]:
mesh

In [ ]:
mesh  # EQIORA_THIRD_MAIN_VIEW

In [ ]:
# IPython caches every cell result in Out/_oh, _N, _ and DisplayHook._, so a Mesh
# published as a cell result is retained by the host itself for the life of the
# kernel. display() takes the identical Mesh._repr_mimebundle_ path and builds
# the identical widget view without entering that cache, which is what lets the
# next cell observe Eqiora's own retention alone.
temporary_mesh = make_mesh()
temporary_mesh_reference = weakref.ref(temporary_mesh)
display(temporary_mesh)
temporary_model_id = current_delegate_model_id(temporary_mesh)
del temporary_mesh

In [ ]:
# Claim: displaying a Mesh adds no Eqiora-owned strong reference to it. The
# widget delegate outlives this cell inside ipywidgets' global registry, so a
# delegate, comm, or Eqiora module that retained the Mesh fails this assertion.
# Nothing is claimed here about a Mesh that the host's own caches still hold.
gc.collect()
assert temporary_mesh_reference() is None
print('EQIORA_TEMPORARY_MESH_COLLECTED')

In [ ]:
# EQIORA_CLOSE_MAIN_TRIGGER - kernel-side close affordance
# (`interfaces.python-rich-mesh-display` evidence O7). The initial Run All
# Cells arms it; each later host-driven rerun closes
# the accepted Mesh's current delegate, named only from the kernel-held Mesh
# object. No browser-supplied identifier is read, so the same trigger also
# closes a fresh delegate created by a later redisplay.
main_close_runs += 1
if main_close_runs > 1:
    close_delegate(current_delegate_model_id(mesh))
    print('EQIORA_MAIN_DELEGATE_CLOSED', flush=True)

In [ ]:
# EQIORA_CLOSE_TEMPORARY_TRIGGER - kernel-side close affordance
# (`interfaces.python-rich-mesh-display` evidence O7). The target is the
# model id this notebook recorded at display time;
# the delegate outlives the collected Mesh in ipywidgets' registry.
temporary_close_runs += 1
if temporary_close_runs > 1:
    close_delegate(temporary_model_id)
    print('EQIORA_TEMPORARY_DELEGATE_CLOSED', flush=True)

In [ ]:
assert_unchanged(mesh, accepted_snapshot)
print('EQIORA_MESH_UNCHANGED')